# Tokenization And Cost

Tokenization is the first practical bottleneck in LLM application design. Every prompt, retrieved document, conversation turn, and generated answer is converted into tokens. These tokens determine fit, cost, and often latency. This notebook turns these constraints from invisible limits to plannable quantities.


## Learning Objectives

By the end of this notebook, you should be able to:

- Compare words and approximate token counts.
- Estimate request cost from input and output token counts.
- Build a context-window budget.
- Decide what to shorten or retrieve when a prompt exceeds the context window.


## The Token Constraint

Tokenization is where language becomes an operational constraint. A prompt is not only text; it is billable input, context-window pressure, latency, and reserved space for the answer. Product teams often feel token limits when adding examples, long chat history, or retrieved documents to a prompt.

This notebook keeps the tokenizer approximate so the mechanics stay visible. The exact counts differ by provider and model, but the budgeting habit transfers directly to production systems.

**Tokens are the unit of operation.**

LLM systems do not process text as sentences or human-visible words. They process model-specific tokens. A token may be a full word, a word fragment, a punctuation mark, a number fragment, a whitespace-sensitive marker, or a special control token. This matters because context limits, latency, and pricing are usually measured in tokens rather than words.

Tokenization is also why structured prompts can become expensive quickly. JSON, code, URLs, tables, and repeated examples often contain many punctuation and separator tokens. A prompt that looks short to a human can be large to a model. A practical engineer therefore checks token pressure before adding more retrieved context, examples, or verbose instructions.

To learn how token calculation and pricing are implemented programmatically on API-driven platforms, see the [OpenAI Cookbook: How to Count Tokens with Tiktoken](https://cookbook.openai.com/examples/how_to_count_tokens_with_tiktoken) and the [Hugging Face Tokenizers Guide](https://huggingface.co/docs/tokenizers/index). To count tokens programmatically, see the [tiktoken GitHub Repository](https://github.com/openai/tiktoken). For the theoretical foundations of subword algorithms, see the original BPE paper: [Neural Machine Translation of Rare Words with Subword Units (Sennrich et al., 2015)](https://arxiv.org/abs/1508.07909), and explore Hugging Face's [BPE Tokenization Guide](https://huggingface.co/docs/tokenizers/components#models).


## Context Windows As Capacity Planning

A context window is a fixed-capacity workspace. System instructions, examples, conversation history, retrieved documents, the current user question, and the generated answer all compete for the same capacity. The safe workflow is to reserve output space first, then decide how much retrieved context and history can fit. If the remaining budget is negative, the prompt is too large before generation even starts.


## Why Token Budgets Are Product Decisions

Token counts affect more than billing. They influence latency, context-window fit, retrieval design, and the amount of conversation history an application can keep. A prompt that is cheap during testing can become expensive when it runs for many users, many documents, or many turns.

For that reason, token budgeting belongs early in application design. Before choosing a model, decide what must be in the prompt, what can be shortened, what can be retrieved only when needed, and how much output space the answer requires.


## Implement The Tokenizer

The next helper cell defines a transparent approximation of token counting and cost estimation. The tokenizer is deliberately simple: it uses a regular expression so you can inspect exactly how text is split. The budget helper then turns a raw token count into a context-window decision: how much room is left after prompt text and reserved output space?

Production systems should use the tokenizer for the target model, but the engineering workflow is the same: tokenize input, count expected output, reserve context space, and estimate cost from provider pricing.


```mermaid
graph LR
    A[Prompt text] --> B[Tokenize]
    B --> C[Count input tokens]
    C --> D[Reserve output tokens]
    D --> E[Check context window]
    E --> F[Estimate cost]
```


In [ ]:
# These helper functions are intentionally approximate — production tokenizers are model-specific.
import re
from dataclasses import dataclass

TOKEN_PATTERN = re.compile(
    r"""
    [A-Za-z]+(?:'[A-Za-z]+)?  # Words and contractions
    |\d+(?:\.\d+)?            # Integers and decimals
    |[^\w\s]                  # Punctuation and symbols
    """,
    re.VERBOSE,
)


@dataclass(frozen=True)
class TokenizationResult:
    text: str
    tokens: list[str]

    @property
    def count(self) -> int:
        return len(self.tokens)


def simple_tokenize(text: str) -> list[str]:
    """Split text into approximate tokens."""
    return TOKEN_PATTERN.findall(text)


def inspect_text(text: str) -> TokenizationResult:
    return TokenizationResult(text=text, tokens=simple_tokenize(text))


def count_tokens(text: str) -> int:
    return len(simple_tokenize(text))


def token_budget_remaining(
    context_window: int, prompt_tokens: int, reserved_output_tokens: int
) -> int:
    return context_window - prompt_tokens - reserved_output_tokens

Cost estimation follows directly from token counting: once you know how many input and output tokens a request uses, you multiply by the provider's per-token price.


In [ ]:
# Cost helpers are kept separate so input and output pricing remain inspectable.
@dataclass(frozen=True)
class CostEstimate:
    input_tokens: int
    output_tokens: int
    input_cost: float
    output_cost: float

    @property
    def total_cost(self) -> float:
        return self.input_cost + self.output_cost


def estimate_request_cost(
    input_tokens: int,
    output_tokens: int,
    input_price_per_million: float,
    output_price_per_million: float,
) -> CostEstimate:
    input_cost = input_tokens / 1_000_000 * input_price_per_million
    output_cost = output_tokens / 1_000_000 * output_price_per_million
    return CostEstimate(input_tokens, output_tokens, input_cost, output_cost)


def format_usd(value: float) -> str:
    """Format small request costs consistently for comparison tables."""
    return f"${value:.6f}"

## Tokenization Is Not Word Splitting

Production tokenizers use model-specific vocabularies, typically built using algorithms like Byte-Pair Encoding (BPE). BPE builds a vocabulary iteratively by starting with character-level tokens and merging the most frequent adjacent token pairs, forming a merge tree or hierarchy (illustrated below) that allows the model to handle both common words and unseen words efficiently. This lesson uses a simple local tokenizer so you can inspect punctuation, URLs, JSON and code without an API.

<div align="center">
  <img src="assets/02-bpe-tokenization.png" alt="Subword Tokenization (BPE)" width="600" style="max-width: 100%; height: auto;" />
  <p><em>Byte-Pair Encoding subword tokenization merge tree.</em></p>
</div>


The next cell sends several text formats through our tokenizer. Notice the pattern: natural language usually tokenizes compactly, while JSON, URLs, and code often add many punctuation or separator tokens. That observation helps you decide where prompt compression is most valuable.


In [ ]:
# Compare text shapes that often tokenize differently: prose, JSON, URLs, and code.
examples = [
    "Explain why token counts matter for LLM applications.",
    "Return JSON with keys: category, priority, rationale.",
    '{"question": "How much will this prompt cost?", "max_tokens": 200}',
    "https://developers.openai.com/api/docs/models",
    "def estimate_cost(tokens): return tokens / 1_000_000 * 0.50",
]
for text in examples:
    result = inspect_text(text)
    print(f"{result.count:>2} tokens | {result.text}")
    print("   ", result.tokens)
    print()

## Cost Estimation

Hosted LLM APIs often price input and output tokens separately. Even tiny per-request costs become meaningful at product scale.


### Prompt Budget Flow

```mermaid
graph LR
    A["System instructions"] --> T["Total input tokens"]
    B["Examples"] --> T
    C["Conversation history"] --> T
    D["Retrieved chunks"] --> T
    E["User question"] --> T
    T --> W{"Fits context window?"}
    W -- Yes --> C1["Reserve output tokens"]
    W -- No --> R["Trim, summarize,<br/>or retrieve less"]
    C1 --> P["Estimate price<br/>and latency"]
```

The important habit is to reserve output tokens before filling the context window. If the input consumes all available tokens, the model has no room left to answer.


In [ ]:
# Estimate cost for the examples above and show how it scales with request volume.
prompt_tokens = sum(inspect_text(text).count for text in examples)
reserved_output_tokens = 250
estimate = estimate_request_cost(
    input_tokens=prompt_tokens,
    output_tokens=reserved_output_tokens,
    input_price_per_million=0.50,
    output_price_per_million=1.50,
)
print(f"Input tokens: {estimate.input_tokens}")
print(f"Output tokens: {estimate.output_tokens}")
print(f"Total cost per request: {format_usd(estimate.total_cost)}")
print(f"Total for 1,000 requests: {format_usd(estimate.total_cost * 1_000)}")
print(f"Total for 100,000 requests: {format_usd(estimate.total_cost * 100_000)}")

### Context Budget

A context window includes system instructions, examples, conversation history, retrieved documents, user input, and generated output. Always reserve output space.


In [ ]:
# Reserve answer space first, then calculate how much room remains for retrieval.
context_window = 16_000
system_and_examples = 2_000
user_question = 250
reserved_answer = 1_500
retrieved_chunk_size = 700
remaining_for_retrieval = token_budget_remaining(
    context_window=context_window,
    prompt_tokens=system_and_examples + user_question,
    reserved_output_tokens=reserved_answer,
)
chunk_count = remaining_for_retrieval // retrieved_chunk_size
print(f"Remaining tokens for retrieved context: {remaining_for_retrieval}")
print(f"Approximate 700-token chunks that fit: {chunk_count}")

## Product Scenarios

Use the table below to compare low-volume and high-volume use cases. The input and output prices are examples, not recommendations.


### Cost Scales With Product Design

The price of one request can look negligible, but cost becomes a design constraint when multiplied by traffic, retries, long outputs, and retrieved context. A RAG answer may cost more than a simple classification because it includes source passages. A summarization workflow may cost more because output length is high. Token cost should therefore be estimated during feature design, not after launch.


In [ ]:
# Compare usage scenarios with the same cost formula to make scale effects visible.
scenarios = [
    ("Simple extraction", 600, 120, 0.50, 1.50),
    ("RAG answer", 4_500, 800, 0.50, 1.50),
    ("Long report summary", 22_000, 1_500, 1.25, 5.00),
]

header = f"{'Scenario':<22} {'Per request':>12} {'Per 100k requests':>18}"
print(header)
print("-" * len(header))

for name, input_tokens, output_tokens, input_price, output_price in scenarios:
    cost = estimate_request_cost(
        input_tokens,
        output_tokens,
        input_price,
        output_price,
    )
    print(
        f"{name:<22} {format_usd(cost.total_cost):>12} {f'${cost.total_cost * 100_000:.2f}':>18}"
    )

Notice that the third scenario uses higher per-token prices because long summarization tasks typically require a more capable, more expensive model. Cost differences here reflect both token volume and model tier combined.

### Interpreting The Cost Table

The cost scenarios are deliberately simple, but they reflect a real engineering tradeoff. A RAG answer often costs more than simple extraction because retrieved context becomes input tokens. A long report summary can be expensive even before considering latency. A high-volume support bot may have a small per-request cost but a meaningful monthly bill.

When comparing prompt designs, record the expected input tokens, expected output tokens and volume assumption together. A cheap prompt at 100 requests can be a serious cost at 1,000,000 requests.


## What To Inspect

Look for inputs that are surprisingly token-heavy: stale conversation history, oversized retrieved chunks, repeated examples, verbose formatting instructions, URLs, JSON, and code snippets. Most of these can be shortened without removing what the model needs to complete the task. Cutting core instructions or directly relevant context reduces cost but tends to degrade the answer.


## Common Mistakes

- Counting words instead of tokens.
- Forgetting that retrieved documents and conversation history are input tokens.
- Asking for a long answer without reserving output space.
- Repeating long static instructions in every call.
- Reducing context so aggressively that the answer no longer has enough evidence.

A good prompt budget is not just shorter. It is shorter while still preserving the information needed for the task.


## Practice

Edit `your_prompt` and rerun the cell. Try a support question, a JSON input, and a long instruction with bullet points. Notice how token count changes across prompt styles.


In [ ]:
# @TODO: Change `your_prompt` and rerun the cell to inspect token pressure.

your_prompt = "Classify this support question and return one sentence of rationale."

result = inspect_text(your_prompt)
print(f"Approximate token count: {result.count}")
print(result.tokens)

### Practice Reference

A support question usually has a modest token count, a JSON input grows quickly because punctuation and keys become tokens, and a long instruction with many bullets increases both cost and context pressure. What matters, though, is not the number itself but where it came from and what is safe to cut without removing task-critical evidence.

<details>
  <summary>Show Examples</summary>

  1. Support question:
   
     ```python
     your_prompt = "My payment failed. Can you help?"
     ```

  2. JSON:
   
     ```python
     your_prompt = '{"customer_id": "8821", "name": "Sven Young", "plan": "pro", "open_tickets": 3, \
      "last_contact": "2026-05-10", "issue": "Payment failed but account still charged"}'
     ```

  3. Long instruction with bullet points:
   
     ```python
     your_prompt = "You are a customer support assistant for an e-commerce platform. When responding \
      to a customer message, follow these instructions: \
      - Greet the customer by name if their name appears in the input. \
      - Classify the issue into one of four categories: billing, shipping, technical, or general inquiry. \
      - Assign a priority: high if the customer mentions urgency, frustration, or financial loss; \
        medium for standard requests; low for general questions. \
      - Propose one concrete next step the customer can act on immediately. \
      - Limit your response to 100 words or fewer. \
      - Never promise a refund, credit, or replacement without first escalating to a human agent. \
      - If the issue is unclear, ask one clarifying question before proceeding."
     ```
     
  </details>


## Check Your Understanding

- Why is token count different from word count?
- What consumes the context window?
- What would you shorten first in an expensive prompt?
- Why should output space be reserved?

<details>
<summary>Show Reference Answers</summary>

- Token count differs from word count because tokenizers split text into model-specific units such as subwords, punctuation, symbols, and sometimes whole words.
- System instructions, examples, chat history, retrieved context, user input, and generated output all consume the context window.
- First shorten repeated examples, long retrieved chunks, stale conversation history, and verbose formatting instructions while preserving the evidence needed for the task.
- Output space must be reserved because the model needs room to generate the answer; a prompt that fills the full context window can fail or truncate useful output.

</details>
